In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Admin\AppData\Local\Temp\ipykernel_22692\959662780.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [ ]:


client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENAI_BASE_URL)

llm = ChatOpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENAI_BASE_URL, model=MODEL, temperature=0.2)
embedding = OpenAIEmbeddings(api_key=OPENROUTER_API_KEY, base_url=OPENAI_BASE_URL, model=EMBED_MODEL)

In [3]:
def load_document(file_path: str):
    """Load a document using the right LangChain loader based on extension."""
    if file_path.lower().endswith('pdf'):
        loader = PyPDFLoader(file_path)
    elif file_path.lower().endswith('txt'):
        loader = TextLoader(file_path, encoding='utf-8')
    else:
        raise ValueError(f"Unsupported file type: {file_path}")

    documents = loader.load()
    return documents

In [4]:
def chunk_documents(documents, chunk_size: int=500, chunK_overlap: int=50):
    """Split loaded documents into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunK_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks = splitter.split_documents(documents)
    return chunks

In [5]:
docs = load_document('data/sample.txt')
chunks = chunk_documents(docs)

print(f"Loaded {len(docs)} document(s), split into {len(chunks)} chunks.")
print("\nFirst chunk content:\n", chunks[0].page_content)
print("\nFirst chunk metadata:\n", chunks[0].metadata)

Loaded 1 document(s), split into 5 chunks.

First chunk content:
 Artificial intelligence has transformed the way businesses operate over the last decade. Companies across industries are now using machine learning models to automate repetitive tasks, predict customer behavior, and optimize supply chains. What once required teams of analysts can now be done in seconds by well-trained algorithms.

First chunk metadata:
 {'source': 'data/sample.txt'}


In [6]:
# Dekho ki chunks kaise dikhte hain, overlap kaise kaam kar raha hai
for i, c in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content)
    print()

--- Chunk 0 (331 chars) ---
Artificial intelligence has transformed the way businesses operate over the last decade. Companies across industries are now using machine learning models to automate repetitive tasks, predict customer behavior, and optimize supply chains. What once required teams of analysts can now be done in seconds by well-trained algorithms.

--- Chunk 1 (392 chars) ---
Retrieval-Augmented Generation, or RAG, is one of the most practical applications of large language models today. Instead of relying purely on what a model learned during training, RAG systems fetch relevant information from an external knowledge base at query time. This allows the model to answer questions using up-to-date or domain-specific information it was never explicitly trained on.

--- Chunk 2 (472 chars) ---
A typical RAG pipeline has three main stages. First, documents are broken into smaller chunks and converted into vector embeddings, which are numerical representations that capture semantic

In [7]:
vectortstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory='chroma_db',
    collection_name='rag_docs'
)

In [8]:
query = "What are the main stages of a RAG pipeline?"
results = vectortstore.similarity_search(query, k=2)  # top 2 most relevant chunks

for i, r in enumerate(results):
    print(f"--- Result {i} ---")
    print(r.page_content)
    print(r.metadata)
    print()

--- Result 0 ---
A typical RAG pipeline has three main stages. First, documents are broken into smaller chunks and converted into vector embeddings, which are numerical representations that capture semantic meaning. Second, these embeddings are stored in a vector database for fast similarity search. Third, when a user asks a question, the system retrieves the most relevant chunks and passes them to the language model as context, which then generates an answer grounded in that context.
{'source': 'data/sample.txt'}

--- Result 1 ---
However, RAG systems are not without challenges. The quality of retrieval directly affects the quality of the final answer. If the chunking strategy is poor, or if the embedding model fails to capture the right semantic relationships, the system may retrieve irrelevant context, leading to hallucinated or incorrect responses. This is why careful design of chunk size, overlap, and retrieval strategy matters so much in building a reliable RAG pipeline.
{'source

In [9]:
def rag_answer(query: str, k: int=3):
    results = vectortstore.similarity_search(query, k=k)

    context = "\n\n".join([r.page_content for r in results])

    prompt = f"""Answer the question based only on the context below.
             If the answer isn't in the context, say "I don't have enough information to answer that."

            Context:
            {context}

            Question: {query}

            Answer:"""

    response = llm.invoke(prompt)
    return response, results

In [12]:
answer, sources = rag_answer("What are the main stages of a RAG pipeline?")
print("ANSWER:\n", answer)
print("\n--- SOURCES USED ---")
for s in sources:
    print("-", s.page_content[:80], "...")

ANSWER:
 content='The main stages of a RAG pipeline are: \n1. Documents are broken into smaller chunks and converted into vector embeddings.\n2. These embeddings are stored in a vector database for fast similarity search.\n3. When a user asks a question, the system retrieves the most relevant chunks and passes them to the language model as context to generate an answer.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 302, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 8.79e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 8.79e-05, 'upstream_inference_prompt_cost': 4.53e-05, 'upstream_infer

In [11]:
answer, sources = rag_answer("What is the capital of France?")
print("ANSWER:\n", answer)

ANSWER:
 content="I don't have enough information to answer that." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 285, 'total_tokens': 295, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 4.875e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 4.875e-05, 'upstream_inference_prompt_cost': 4.275e-05, 'upstream_inference_completions_cost': 6e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_369e662417', 'id': 'gen-1787206205-YweffmD8eWmfohQeMjw7', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a01dca-6fd8-76d0-80cf-7f6808bfeb88-0' tool_calls=[] invalid_tool_c